# Code agent with DuckDuck search tool

In [ ]:
from huggingface_hub import login

login()

from smolagents import CodeAgent, DuckDuckGoSearchTool, InferenceClientModel

agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct"))

agent.run("Search for the best music recommendations for a party at the Wayne's mansion. ")

# Simple Custom Tool

In [ ]:
from smolagents import CodeAgent, tool, InferenceClientModel

@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion (str): The type of occasion for the party. Allowed values are:
                        - "casual": Menu for casual party.
                        - "formal": Menu for formal party.
                        - "superhero": Menu for superhero party.
                        - "custom": Custom menu.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

agent = CodeAgent(tools=[suggest_menu], model=InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct"), additional_authorized_imports=["sys"])
agent.run("Prepare a formal menu for the party.")

# Authorizing import 

In [ ]:
from smolagents import CodeAgent, InferenceClientModel
import numpy as np
import time
import datetime

agent = CodeAgent(tools=[], model=InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct"), additional_authorized_imports=['datetime'])

agent.run(
    """
    Alfred needs to prepare for the party. Here are the tasks:
    1. Prepare the drinks - 30 minutes
    2. Decorate the mansion - 60 minutes
    3. Set up the menu - 45 minutes
    4. Prepare the music and playlist - 45 minutes

    If we start right now, at what time will the party be ready?
    """
)

# All Combined

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, FinalAnswerTool, InferenceClientModel, Tool, tool, VisitWebpageTool

@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion: The type of occasion for the party.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.
    
    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }
    
    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)
    
    return best_service

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""
    
    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }
    
    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }
        
        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")


# Alfred, the butler, preparing the menu for the party
agent = CodeAgent(
    tools=[
        DuckDuckGoSearchTool(), 
        VisitWebpageTool(),
        suggest_menu,
        catering_service_tool,
        SuperheroPartyThemeTool(),
	FinalAnswerTool()
    ], 
    model=InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct"),
    max_steps=10,
    verbosity_level=2
)

agent.run("Give me the best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")

# Inspecting the agent with OpenTelemetry and Langfuse

In [6]:
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd().parent / "Langfuse" / ".env"
load_dotenv(dotenv_path=env_path)
print(env_path, env_path.exists())

/home/nerfeddoge/Documents/Personal/AI Agent course - Hugging face/Langfuse/.env True


In [7]:
from langfuse import get_client
 
langfuse = get_client()
 
# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

Langfuse client is authenticated and ready!


In [8]:
from openinference.instrumentation.smolagents import SmolagentsInstrumentor

SmolagentsInstrumentor().instrument()

In [10]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, FinalAnswerTool, InferenceClientModel, Tool, tool, VisitWebpageTool

@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion: The type of occasion for the party.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.
    
    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }
    
    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)
    
    return best_service

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""
    
    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }
    
    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }
        
        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")


# Alfred, the butler, preparing the menu for the party
agent = CodeAgent(
    tools=[
        DuckDuckGoSearchTool(), 
        VisitWebpageTool(),
        suggest_menu,
        catering_service_tool,
        SuperheroPartyThemeTool(),
	FinalAnswerTool()
    ], 
    model=InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct"),
    max_steps=10,
    verbosity_level=2
)

agent.run("Give me the best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me the best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme    │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: To generate the best playlist for a 'villain masquerade' themed party at Wayne's mansion, I will first    
request a specific superhero-party theme for villains. Then, I'll use this theme to guide the selection of music   
that matches the atmosphere of the event.                                                                          
                                                                                                                   
<code>                                                                                                             
# Requesting the villain masquerade theme for inspiration                                                          
theme = superhero_party_theme_generator(category='villain masquerade')                                             
print(f"The suggested theme for the 'villain masquerade' party is: {theme}")                                       
                                                                                                                   
# Suggesting a menu for a villain masquerade party (though it's not strictly necessary, we can take the theme into 
consideration)                                                                                                     
menu_suggestion = suggest_menu(occasion='Villain masquerade party')                                                
print(f"The suggested menu for the party is: {menu_suggestion}")                                                   
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Requesting the villain masquerade theme for inspiration                                                        
  theme = superhero_party_theme_generator(category='villain masquerade')                                           
  print(f"The suggested theme for the 'villain masquerade' party is: {theme}")                                     
                                                                                                                   
  # Suggesting a menu for a villain masquerade party (though it's not strictly necessary, we can take the theme    
  into consideration)                                                                                              
  menu_suggestion = suggest_menu(occasion='Villain masquerade party')                                              
  print(f"The suggested menu for the party is: {menu_suggestion}")                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
The suggested theme for the 'villain masquerade' party is: Gotham Rogues' Ball: A mysterious masquerade where 
guests dress as classic Batman villains.
The suggested menu for the party is: Custom menu for the butler.

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: Based on the web search results, two potential playlists stand out for their focus on classic Batman      
villains and appropriate music for a masquerade theme. One is on YouTube, and the other is on Spotify. Since       
visiting YouTube URLs may not be directly actionable within this format, and Spotify links are more accessible, I  
will select the Spotify playlist.                                                                                  
                                                                                                                   
I will choose "Gotham Villains Meeting | Batman 90s Vibes | Noir Jazz Mix" because it contains 50 items, which     
should provide a good variety of music to set the mood for the party.                                              
                                                                                                                   
<code>                                                                                                             
# Printing the selected Spotify playlist URL for reference                                                         
playlist_url = "https://open.spotify.com/playlist/0IJp7u8Cmyg3GkAu6tObt7"                                          
final_answer(playlist_url)                                                                                         
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Printing the selected Spotify playlist URL for reference                                                       
  playlist_url = "https://open.spotify.com/playlist/0IJp7u8Cmyg3GkAu6tObt7"                                        
  final_answer(playlist_url)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: https://open.spotify.com/playlist/0IJp7u8Cmyg3GkAu6tObt7

[Step 3: Duration 4.40 seconds| Input tokens: 8,805 | Output tokens: 451]

'https://open.spotify.com/playlist/0IJp7u8Cmyg3GkAu6tObt7'